# Joint M2 + ACT fit vs M2 only: grid stability and dataset consistency

Why the MOOJ1142 result to quote is the joint M2 + ACT fit rather than the M2 only fit, quantitatively.

**Method**
* **Grid stability and convergence:** the same cluster model (gNFW_rs + isobeta subcluster + point source) fit to M2
  alone and jointly with ACT on model grids of different size (`coords.r_map`). Each run is loaded from its output
  directory with `witch.model_comparison.load_run`. A result that moves when the grid grows depends on the line of sight
  truncation rather than the data; R-hat says whether the chains agree.
* **Consistency** (from `scripts/joint_consistency.sbatch`, which reloads the joint run's data under MPI; nothing here
  loads the data, so the notebook runs anywhere in seconds). Both posteriors are scored on the same data, the joint
  run's M2 TODs and ACT map, under a **fixed noise model** per dataset:
  * ACT: ivar + CMB, it doesn't depend on any model, so ACT chi-squared are directly comparable.
  * M2: the noise is estimated from the TODs, so it has to be fixed to compare two parameter sets. Each choice has a
    catch, so all three are shown:
    * `joint_median` / `m2_median`: from the residual of that run's MCMC medians, as the fits do. With the right model
      subtracted this is just the noise, but between two models it favours its reference: the reference's misfit in
      the other's residual is counted as noise and down weighted.
    * `initial`: from the data with no model subtracted (as `aic_refit`). Model independent, but it contains the
      cluster, down weights the cluster's modes, and so moves the M2 best fit far from both posteriors.
* **Posterior predictive:** chi-squared of 300 draws from each posterior on each dataset separately, relative to that
  dataset's own minimum. The ACT point source amplitude isn't in the M2 only model and enters linearly, so it is set to
  its best value (within its prior) for every draw of both posteriors (`witch.model_comparison.profile_linear`). Draws
  from a posterior the data agree with sit about `N_eff` (the number of parameters the data constrain) above the
  minimum.
* **Q_DMAP** (Raveri & Hu 2019): `Q = chi2_joint,min - chi2_M2,min - chi2_ACT,min`, the sum of what the joint fit costs
  each dataset, against a chi-squared with `N_eff(M2) + N_eff(ACT) - N_eff(joint)` dof (`witch.model_comparison.q_dmap`,
  `n_eff` from each dataset's Fisher matrix with the flat priors as Gaussians of the same variance). A small Q means the
  datasets are consistent and combining them is justified.

In [ ]:
REDUCTIONS = "/scratch/jorlo/MUSTANG/Reductions/MOOJ1142"
M2_DIR = f"{REDUCTIONS}/mustang2/cluster_model_sph_isobeta-gnfw_rs+ps_model_ps_gauss"
JOINT_DIR = f"{REDUCTIONS}/mustang2+act/cluster_model_sph_isobeta-gnfw_rs+ps_model_ps_gauss+ps_act_ps_gauss"
M2_PARS = "r-beta-amp-P0-r_s-beta-dx_g-dy_g-sigma-amp_g"
JOINT_PARS = f"{M2_PARS}-amp_g"
# The same model on different grids, label -> output directory. All start from the M2 only MH medians.
RUNS = {
    "M2 3'": f"{M2_DIR}/xray_peak_rs_control_rmap3/{M2_PARS}",  # 4 x 10000 MH steps
    "M2 7'": f"{M2_DIR}/xray_peak_rs_control_rmap7/{M2_PARS}",  # 4 x 10000
    "M2 10'": f"{M2_DIR}/xray_peak_rs_control_rmap10/{M2_PARS}",  # 4 x 10000
    "joint 7'": f"{JOINT_DIR}/joint_needlet_mcmc_3as/{JOINT_PARS}",  # 4 x 40000
    "joint 10'": f"{JOINT_DIR}/joint_mcmc_rmap10/{JOINT_PARS}",  # 4 x 40000
}
# Grid changes to check, (smaller, larger): shifts are in units of the larger grid's posterior width
GRID_PAIRS = [("M2 3'", "M2 7'"), ("M2 7'", "M2 10'"), ("joint 7'", "joint 10'")]
# scripts/joint_consistency.sbatch output for the joint 10' and M2 only 10' fits, all three M2 noise models
CONSISTENCY = "/scratch/jorlo/joint_consistency_2920370.yaml"
POSTERIORS = {"joint": "joint 10'", "m2": "M2 10'"}  # posterior labels in CONSISTENCY -> RUNS
M2_NAME, ACT_NAME = "mustang2", "act"  # dataset names
MAX_RHAT = 1.1
FIGDIR = "figures"


In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import display
from scipy import stats

os.environ.setdefault("JAX_PLATFORMS", "cpu")  # before witch imports jax
os.makedirs(FIGDIR, exist_ok=True)

from witch import model_comparison as wmc  # noqa: E402

pd.set_option("display.float_format", lambda x: f"{x:.4g}")


def half_width(x: np.ndarray) -> float:
    # Half the 16-84 percentile range, the posterior 'sigma' used throughout
    return float(np.diff(np.percentile(x, [16, 84]))[0] / 2)


In [ ]:
runs = {label: wmc.load_run(outdir, label) for label, outdir in RUNS.items()}
display(pd.DataFrame([
    {"run": r.label, "r_map": r.cfg["coords"]["r_map"], "k": r.k, "chains": r.chains.shape, "outdir": r.outdir}
    for r in runs.values()
]))
# Free parameters shared by every run (the ACT point source amplitude is only in the joint fits)
shared = [n for n in next(iter(runs.values())).free_names if all(n in r.free_names for r in runs.values())]
cluster = [n for n in shared if n.startswith("cluster_model.")]


## Chain convergence

Split R-hat above ~1.1 means the chains disagree, so that run's medians and widths aren't reliable. A run that only
fails in one direction of parameter space (here the gNFW P0 / r_s / beta degeneracy) is one whose data don't constrain
that direction.

In [ ]:
diag = pd.DataFrame([row for run in runs.values() for row in wmc.chain_diagnostics(run)])
display(diag.pivot(index="parameter", columns="run", values="Rhat")[list(runs)].rename_axis("Rhat", axis=1))
for label, worst in diag.groupby("run", sort=False)["Rhat"].max().items():
    if worst > MAX_RHAT:
        print(f"WARNING {label} is not converged, max Rhat {worst:.3f}")


## Grid stability

Posterior median ± half the 16-84% range of the shared free parameters, then the shift of each median when the grid
grows, in units of the larger grid's width. Shifts well under 1 mean the result has converged in grid size.

In [ ]:
post = {}
for label, run in runs.items():
    flat = run.chains.reshape(-1, run.k)
    post[label] = {
        n.split(".", 1)[1]: (np.median(flat[:, run.free_names.index(n)]), half_width(flat[:, run.free_names.index(n)]))
        for n in shared
    }
display(pd.DataFrame({label: {p: f"{m:.4g} ± {s:.2g}" for p, (m, s) in v.items()} for label, v in post.items()}))
display(pd.DataFrame({
    f"{a} -> {b}": {p: (post[b][p][0] - post[a][p][0]) / post[b][p][1] for p in post[a]} for a, b in GRID_PAIRS
}).rename_axis("shift / sigma"))


In [ ]:
ncol = 3
fig, axes = plt.subplots(int(np.ceil(len(cluster) / ncol)), ncol, figsize=(4 * ncol, 3 * np.ceil(len(cluster) / ncol)))
for ax, name in zip(axes.flat, cluster):
    for label, run in runs.items():
        ax.hist(run.chains[..., run.free_names.index(name)].ravel(), bins=60, density=True, histtype="step",
                ls="-" if label.startswith("joint") else "--", label=label)
    ax.set_xlabel(name.split(".", 1)[1])
for ax in list(axes.flat)[len(cluster):]:
    ax.set_visible(False)
axes.flat[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, "joint_grid_posteriors.png"), dpi=150)


## Posterior predictive: each posterior on each dataset

`dchisq` is relative to that dataset's own minimum under the same noise. Per M2 noise model:
* `M2 prefers M2-only by` = M2 dchisq(joint median) - M2 dchisq(M2 only median).
* `ACT penalty on M2-only` = ACT dchisq(M2 only median) - ACT dchisq(joint median). The same for every M2 noise model.
* `combined prefers joint by` = the difference of the two, the total chi-squared difference between the two solutions
  (same number of parameters, the ACT amplitude being set to its best value for the M2 only one).

In [ ]:
with open(CONSISTENCY) as f:
    cons = yaml.safe_load(f)
draws = np.load(os.path.splitext(CONSISTENCY)[0] + "_draws.npz")
for key, label in POSTERIORS.items():
    if os.path.abspath(cons["posteriors"][key]) != os.path.abspath(runs[label].outdir):
        raise ValueError(f"{CONSISTENCY} used {cons['posteriors'][key]} for {key}, not {label}")
noises = [k for k in cons if isinstance(cons[k], dict) and "Q_DMAP" in cons[k]]
m2k, actk = f"dchisq_{M2_NAME}", f"dchisq_{ACT_NAME}"

rows = []
for noise in noises:
    med = cons[noise]["medians"]
    m2_pref = med["joint_median"][m2k] - med["m2_median"][m2k]
    act_pen = med["m2_median"][actk] - med["joint_median"][actk]
    rows.append({
        "M2 noise": noise,
        "M2: joint median": med["joint_median"][m2k],
        "M2: M2-only median": med["m2_median"][m2k],
        "M2 prefers M2-only by": m2_pref,
        "ACT: joint median": med["joint_median"][actk],
        "ACT: M2-only median": med["m2_median"][actk],
        "ACT penalty on M2-only": act_pen,
        "combined prefers joint by": act_pen - m2_pref,
    })
display(pd.DataFrame(rows).set_index("M2 noise"))

rows = []
for noise in noises:
    for key, d in cons[noise]["posterior_draws"].items():
        for dset in (M2_NAME, ACT_NAME):
            s = d[f"dchisq_{dset}"]
            rows.append({"M2 noise": noise, "posterior": POSTERIORS[key], "dataset": dset, "median": s["median"],
                         "p16": s["p16_p84"][0], "p84": s["p16_p84"][1]})
display(pd.DataFrame(rows).set_index(["M2 noise", "dataset", "posterior"]).sort_index())


### How unlikely is the M2 only solution for ACT?

If ACT agreed with a posterior, its draws would scatter about `N_eff(ACT)` above ACT's minimum, roughly as a
chi-squared with that many dof (the joint draws do). The tail probability of the M2 only median's ACT dchisq under that
distribution, and the fraction of M2 only draws that fit ACT as well as the joint draws' 84th percentile, say how hard
ACT rejects the M2 only solution.

In [ ]:
ref = noises[0]  # ACT doesn't depend on the M2 noise
act_min = cons[ref]["Q_DMAP"]["chisq_min"][ACT_NAME]
neff_act = cons[ref]["Q_DMAP"]["n_eff"][ACT_NAME]
d_act = {key: draws[f"{ref}_{key}"][:, 1] - act_min for key in POSTERIORS}
for key, med in cons[ref]["medians"].items():
    p = stats.chi2.sf(med[actk], neff_act)
    print(f"{key}: ACT dchisq {med[actk]:.1f}, P(chi2_{neff_act:.1f} > that) = {p:.2g} ({stats.norm.isf(p / 2):.2f} sigma)")
print(f"M2 only draws fitting ACT at least as well as the joint draws' 84th percentile: "
      f"{np.mean(d_act['m2'] <= np.percentile(d_act['joint'], 84)):.1%}")

fig, axes = plt.subplots(1, 1 + len(noises), figsize=(4 * (1 + len(noises)), 3.2))
bins = np.linspace(0, np.percentile(d_act["m2"], 97), 40)
for key, label in POSTERIORS.items():
    axes[0].hist(d_act[key], bins=bins, density=True, histtype="step", label=label)
axes[0].plot(bins, stats.chi2.pdf(bins, neff_act), "k:", label=rf"$\chi^2_{{{neff_act:.1f}}}$")
axes[0].set_xlabel(r"ACT $\Delta\chi^2$")
axes[0].legend(fontsize=8)
for ax, noise in zip(axes[1:], noises):
    m2_min = cons[noise]["Q_DMAP"]["chisq_min"][M2_NAME]
    d_m2 = {key: draws[f"{noise}_{key}"][:, 0] - m2_min for key in POSTERIORS}
    b = np.linspace(min(map(np.min, d_m2.values())), np.percentile(np.concatenate(list(d_m2.values())), 98), 40)
    for key, label in POSTERIORS.items():
        ax.hist(d_m2[key], bins=b, density=True, histtype="step", label=label)
    ax.set_xlabel(rf"M2 $\Delta\chi^2$, {noise} noise")
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, "joint_consistency_draws.png"), dpi=150)


## Q_DMAP: are M2 and ACT consistent?

The ACT only minimum frees every parameter ACT sees, including the point source position, which ACT can't place to the
few arcsec M2 does. If that minimum moved the point source far from the joint one it found a CMB or noise feature
instead, which lowers `chi2_ACT,min` and so overestimates `cost_to_act` and Q: then Q is an upper bound.

In [ ]:
display(pd.DataFrame([
    {"M2 noise": noise, **{k: v for k, v in cons[noise]["Q_DMAP"].items() if k in ("Q", "cost_to_m2", "cost_to_act", "dof", "p", "sigma")},
     **{f"N_eff {k}": v for k, v in cons[noise]["Q_DMAP"]["n_eff"].items()}}
    for noise in noises
]).set_index("M2 noise"))


for noise in noises:
    # Minima are keyed by parameter name and index, e.g. dx_g[16]
    joint_min, act_only_min = cons[noise]["minima"]["joint"], cons[noise]["minima"][ACT_NAME]
    pos = [k for k in joint_min if k.startswith(("dx_g[", "dy_g["))]
    offset = np.hypot(*[act_only_min[k] - joint_min[k] for k in pos])
    if offset > 10:
        print(f"WARNING {noise}: the ACT only minimum moved the point source {offset:.0f} arcsec from the joint one, "
              "Q is an upper bound")


## Reading the results
* **Grid stability:** shifts well under 1 sigma between grids mean the fit has converged in grid size. A run that only
  converges on large enough grids was fitting a component whose line of sight integral the small grid truncates.
* **Posterior predictive:** the dataset that decides is the one whose chi-squared separates the two posteriors by much
  more than the other prefers the opposite way. ACT's is noise model independent; M2's preference depends on the noise
  model, and each residual based noise favours its own reference, so read the range across noise models.
* **Q_DMAP:** p of a few percent or more (under ~2 sigma) means no significant tension, so combining the datasets is
  justified and the joint posterior is the better informed one. If the ACT only minimum warning prints, Q is an upper
  bound.

**MOOJ1142, 2026-11-24** (`joint_consistency_2920370`): the joint fit moves ≤ 0.2 sigma from 7' to 10' and converges
(R-hat ≤ 1.007); the M2 only fit's subcluster moves 4-5 sigma from 3' to 7', is stable from 7' to 10', but its gNFW
doesn't converge (R-hat up to 1.2). ACT penalises the M2 only median by dchisq ≈ 18 (P ≈ 5e-4 if ACT agreed with it,
≈ 3.5 sigma), while M2 prefers it by -1 to +6 depending
on the noise model, so the combined data prefer the joint solution by ≈ 12-19. Q is 7.8-14.5 on ≈ 5.2 dof
(1.3-2.4 sigma, an upper bound: the ACT only minimum moved the point source ≈ 3.7'). Quote the joint 10' fit.

To regenerate: `sbatch scripts/joint_consistency.sbatch [<joint outdir> <M2 only outdir>]`, point `CONSISTENCY` at the
new file, and `jupyter nbconvert --to html --execute notebooks/joint_consistency.ipynb` for an HTML report.